# Step 2 - NetDesDuo and NetDes on the synthetic benchmark

Trajectories and decoy-padded initial networks in, per-edge keep/drop tables out.

In [ ]:
import os
import sys

NETDESDUO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "synthetic_data"))
sys.path.insert(0, NETDESDUO_ROOT)
import NetDesDuo
import warnings
import numpy as np
import pandas as pd
import random
import importlib
import math
import joblib
from itertools import permutations
from sklearn.metrics import mean_squared_error
from pathlib import Path

# paths below are relative to ../synthetic_data
os.chdir(DATA_DIR)

## NetDes up to the deletion step

In [ ]:
#runs NetDes completely for one network up to the deletion step
def run_one(seed, num, total_edges):
    base = f"ODE_Case_{seed}"
    data = pd.read_csv(f"{base}/data_{num}.csv", index_col=0)
    
    #make column/cell names + make pseudo 
    pseudo_time = [float(c) for c in data.columns]
    pseudo_time = [(t - min(pseudo_time)) / (max(pseudo_time) - min(pseudo_time)) for t in pseudo_time]
    cell_names = [f"cell_{i}" for i in range(data.shape[1])]
    data.columns = cell_names
    pseudo = pd.DataFrame({"cellnames": cell_names, "Psuedotime": pseudo_time})
    
    #make all interactions to 1
    network = pd.read_csv(f"{base}/total_{total_edges}_edges/initial_network_{num}.csv")
    network = network.iloc[:, :3]
    network.columns = ["Source", "Target", "Interaction"]
    network["Interaction"] = 1
    gene_list = list(data.index.astype(str))
    
    pseudotime, tfpickcell, cellpick = NetDesDuo.selectcell(pseudo, data, pseudotime_col=1)
    scaled_data, log_data, pseudotime_pick = NetDesDuo.process_tf(data, tfpickcell, pseudo, cellpick, pseudotime_col=1)
    
    random.seed(1)
    Part1=NetDesDuo.MSE_stPoint(gene_list=gene_list,network=network, expression=scaled_data,log_expression=log_data)
    print("Part1")
    Part2= NetDesDuo.MSE_top(gene_list=gene_list,network=network, expression=scaled_data,log_expression=log_data,MSE_table=Part1)
    print("Part2")
    w_tot=NetDesDuo.tuning(MSE_table2=Part2,nums=20,gene_list=gene_list)
    Part3=NetDesDuo.MSE_bin(MSE_table2=Part2,w_tot=w_tot,gene_list=gene_list, 
                         network=network,expression=scaled_data,log_expression=log_data)
    consis_results=NetDesDuo.consis_test(gene_list=gene_list,MSE_table=Part3)
    print("Part3")
    int_test=NetDesDuo.interactions_test(gene_list=gene_list,network=network, consis=consis_results,MSE_table=Part3,
                                      expression=scaled_data,log_expression=log_data)
    print("Interactions Test")
    #MSE_cut here doesnt matter
    delete = NetDesDuo.delete_int(gene_list = gene_list,network = network,int_results = int_test,
                               consis_res = consis_results, MSE_cut = 100)
    
    joblib.dump(delete, f"{base}/total_{total_edges}_edges/delete_{num}.joblib")
    joblib.dump(int_test, f"{base}/total_{total_edges}_edges/int_test_{num}.joblib")
    joblib.dump(consis_results, f"{base}/total_{total_edges}_edges/consis_{num}.joblib")
    joblib.dump(Part3, f"{base}/total_{total_edges}_edges/Part3_{num}.joblib")
    return delete, int_test, consis_results

def run_one_folder(seed, total_edges):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"
    
    if not all(Path(f"{out_dir}/{name}_1.joblib").exists() for name in ("delete", "int_test", "consis", "Part3")):
        run_one(seed, 1, total_edges)
    if not all(Path(f"{out_dir}/{name}_2.joblib").exists() for name in ("delete", "int_test", "consis", "Part3")):
        run_one(seed, 2, total_edges)
    return 0

## Run the fits

In [ ]:
#run + cache all the nessecary time consuming netdes steps before deletion
seeds = [125,177,629,753,1419, 1434, 1477,1594,1612,2120, 2569, 3219, 3646, 4119, 4250, 4510, 5042, 5236, 6072, 6534]
total_edges = [27, 30, 33, 36, 39, 42]

tasks = [(s, total_edge) for s in seeds for total_edge in total_edges]

results = joblib.Parallel(n_jobs=15, backend="loky")(
    joblib.delayed(run_one_folder)(s, total_edge) for (s, total_edge) in tasks
)

## Joint deletion across the pair

In [ ]:
def run_delete_combined(seed, total_edges, n_val = 1, k_val = 0.5):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"
    
    network_1 = pd.read_csv(f"{out_dir}/initial_network_1.csv")
    network_2 = pd.read_csv(f"{out_dir}/initial_network_2.csv")
    network_1.columns = ["Source", "Target", "Interaction"]
    network_2.columns = ["Source", "Target", "Interaction"]
    network_1["Interaction"] = 1
    network_2["Interaction"] = 1
    
    genes_1 = list(pd.read_csv(f"{base}/data_1.csv", index_col=0).index.astype(str))
    genes_2 = list(pd.read_csv(f"{base}/data_2.csv", index_col=0).index.astype(str))

    delete_1 = joblib.load(f"{out_dir}/delete_1.joblib")
    delete_2 = joblib.load(f"{out_dir}/delete_2.joblib")

    int_test_1 = joblib.load(f"{out_dir}/int_test_1.joblib")
    int_test_2 = joblib.load(f"{out_dir}/int_test_2.joblib")

    Part3_1 = joblib.load(f"{out_dir}/Part3_1.joblib")
    Part3_2 = joblib.load(f"{out_dir}/Part3_2.joblib")
    
    ob_newall1, ob_newall2 = NetDesDuo.delete_overlapping(delete_1, delete_2, genes_1, genes_2, network_1, network_2, 
                                                int_test_1, int_test_2, Part3_1, Part3_2, n_val, k_val)
    ob_newall1 = NetDesDuo.delete_non_overlapping(delete_1, genes_1, network_1, ob_newall1, n_val)
    ob_newall2 = NetDesDuo.delete_non_overlapping(delete_2, genes_2, network_2, ob_newall2, n_val)

    consis_res_1 = joblib.load(f"{out_dir}/consis_1.joblib")
    consis_res_2 = joblib.load(f"{out_dir}/consis_2.joblib")

    delt_value1, st_new_cut1 = NetDesDuo.make_stpoints(genes_1, network_1, ob_newall1, consis_res_1)
    delt_value2, st_new_cut2 = NetDesDuo.make_stpoints(genes_2, network_2, ob_newall2, consis_res_2)
    
    return ob_newall1, ob_newall2, delt_value1, delt_value2, st_new_cut1, st_new_cut2

def run_delete_combined_signed(seed, total_edges, n_val = 1, k_val = 0.5):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"
    
    network_1 = pd.read_csv(f"{out_dir}/initial_network_1.csv")
    network_2 = pd.read_csv(f"{out_dir}/initial_network_2.csv")
    network_1.columns = ["Source", "Target", "Interaction"]
    network_2.columns = ["Source", "Target", "Interaction"]
    network_1["Interaction"] = 1
    network_2["Interaction"] = 1
    
    genes_1 = list(pd.read_csv(f"{base}/data_1.csv", index_col=0).index.astype(str))
    genes_2 = list(pd.read_csv(f"{base}/data_2.csv", index_col=0).index.astype(str))

    delete_1 = joblib.load(f"{out_dir}/delete_1.joblib")
    delete_2 = joblib.load(f"{out_dir}/delete_2.joblib")

    int_test_1 = joblib.load(f"{out_dir}/int_test_1.joblib")
    int_test_2 = joblib.load(f"{out_dir}/int_test_2.joblib")

    Part3_1 = joblib.load(f"{out_dir}/Part3_1.joblib")
    Part3_2 = joblib.load(f"{out_dir}/Part3_2.joblib")
    
    ob_newall1, ob_newall2 = NetDesDuo.delete_overlapping_signed(delete_1, delete_2, genes_1, genes_2, network_1, network_2, 
                                                int_test_1, int_test_2, Part3_1, Part3_2, n_val, k_val)
    ob_newall1 = NetDesDuo.delete_non_overlapping(delete_1, genes_1, network_1, ob_newall1, n_val)
    ob_newall2 = NetDesDuo.delete_non_overlapping(delete_2, genes_2, network_2, ob_newall2, n_val)

    consis_res_1 = joblib.load(f"{out_dir}/consis_1.joblib")
    consis_res_2 = joblib.load(f"{out_dir}/consis_2.joblib")

    delt_value1, st_new_cut1 = NetDesDuo.make_stpoints(genes_1, network_1, ob_newall1, consis_res_1)
    delt_value2, st_new_cut2 = NetDesDuo.make_stpoints(genes_2, network_2, ob_newall2, consis_res_2)
    
    return ob_newall1, ob_newall2, delt_value1, delt_value2, st_new_cut1, st_new_cut2

def ob_to_edges(gene_list, ob_newall):
    #gene 4 is forced and we delete any remaining interactions
    ob_newall[3] = []
    return {(src, tgt) for tgt, regs in zip(gene_list, ob_newall) for src in regs}

## The AUPRC tables

In [ ]:
def get_row_signs(ob_newall, genes, network, int_test, base_models, initial_edges):
    sign_dict = {}
    for i, gene in enumerate(genes):
        kept = ob_newall[i]
        if len(kept) == 0:
            continue
        signs = NetDesDuo.get_top_sign(genes, gene, network, int_test, kept, base_models)
        for src, is_act in signs.items():
            sign_dict[(src, gene)] = 1 if is_act == 1 else 2
    row = {}
    for (src, tgt) in initial_edges:
        col = f"{src} to {tgt}"
        row[col] = sign_dict.get((src, tgt), 0)
    return row


def export_AUPRC_table(seed, total_edges, n_values, k_val):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"

    genes_1 = list(pd.read_csv(f"{base}/data_1.csv", index_col=0).index.astype(str))
    genes_2 = list(pd.read_csv(f"{base}/data_2.csv", index_col=0).index.astype(str))

    network_1 = pd.read_csv(f"{out_dir}/initial_network_1.csv").iloc[:, :3]
    network_1.columns = ["Source", "Target", "Interaction"]
    network_2 = pd.read_csv(f"{out_dir}/initial_network_2.csv").iloc[:, :3]
    network_2.columns = ["Source", "Target", "Interaction"]

    int_test_1 = joblib.load(f"{out_dir}/int_test_1.joblib")
    int_test_2 = joblib.load(f"{out_dir}/int_test_2.joblib")
    Part3_1 = joblib.load(f"{out_dir}/Part3_1.joblib")
    Part3_2 = joblib.load(f"{out_dir}/Part3_2.joblib")

    initial_edges_1 = list(map(tuple, network_1[["Source", "Target"]].values))
    initial_edges_2 = list(map(tuple, network_2[["Source", "Target"]].values))
    edge_cols_1 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_1]
    edge_cols_2 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_2]

    rows_1 = []
    rows_2 = []
    typed_rows_1 = []
    typed_rows_2 = []

    for n_val in n_values:
        ob1, ob2, delt1, delt2, st1, st2 = run_delete_combined(seed, total_edges, n_val, k_val)

        E1 = ob_to_edges(genes_1, ob1)
        E2 = ob_to_edges(genes_2, ob2)
        E1_set = set(E1)
        E2_set = set(E2)

        row_1 = {"n_val": float(n_val), "k_val": float(k_val)}
        row_2 = {"n_val": float(n_val), "k_val": float(k_val)}
        for (e, col) in zip(initial_edges_1, edge_cols_1):
            row_1[col] = 1 if e in E1_set else 0
        for (e, col) in zip(initial_edges_2, edge_cols_2):
            row_2[col] = 1 if e in E2_set else 0
        rows_1.append(row_1)
        rows_2.append(row_2)

        typed_row_1 = {"n_val": float(n_val), "k_val": float(k_val)}
        typed_row_2 = {"n_val": float(n_val), "k_val": float(k_val)}
        typed_row_1.update(get_row_signs(ob1, genes_1, network_1, int_test_1, Part3_1[5], initial_edges_1))
        typed_row_2.update(get_row_signs(ob2, genes_2, network_2, int_test_2, Part3_2[5], initial_edges_2))
        typed_rows_1.append(typed_row_1)
        typed_rows_2.append(typed_row_2)

    df_1 = pd.DataFrame(rows_1, columns=["n_val", "k_val"] + edge_cols_1)
    df_2 = pd.DataFrame(rows_2, columns=["n_val", "k_val"] + edge_cols_2)
    df_1.to_csv(f"{out_dir}/auprc_table_1.csv", index=False)
    df_2.to_csv(f"{out_dir}/auprc_table_2.csv", index=False)

    tdf_1 = pd.DataFrame(typed_rows_1, columns=["n_val", "k_val"] + edge_cols_1)
    tdf_2 = pd.DataFrame(typed_rows_2, columns=["n_val", "k_val"] + edge_cols_2)
    tdf_1.to_csv(f"{out_dir}/auprc_table_1_typed.csv", index=False)
    tdf_2.to_csv(f"{out_dir}/auprc_table_2_typed.csv", index=False)

    return df_1, df_2, tdf_1, tdf_2


def export_AUPRC_table_signed(seed, total_edges, n_values, k_val):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"

    genes_1 = list(pd.read_csv(f"{base}/data_1.csv", index_col=0).index.astype(str))
    genes_2 = list(pd.read_csv(f"{base}/data_2.csv", index_col=0).index.astype(str))

    network_1 = pd.read_csv(f"{out_dir}/initial_network_1.csv").iloc[:, :3]
    network_1.columns = ["Source", "Target", "Interaction"]
    network_2 = pd.read_csv(f"{out_dir}/initial_network_2.csv").iloc[:, :3]
    network_2.columns = ["Source", "Target", "Interaction"]

    int_test_1 = joblib.load(f"{out_dir}/int_test_1.joblib")
    int_test_2 = joblib.load(f"{out_dir}/int_test_2.joblib")
    Part3_1 = joblib.load(f"{out_dir}/Part3_1.joblib")
    Part3_2 = joblib.load(f"{out_dir}/Part3_2.joblib")

    initial_edges_1 = list(map(tuple, network_1[["Source", "Target"]].values))
    initial_edges_2 = list(map(tuple, network_2[["Source", "Target"]].values))
    edge_cols_1 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_1]
    edge_cols_2 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_2]

    rows_1 = []
    rows_2 = []
    typed_rows_1 = []
    typed_rows_2 = []

    for n_val in n_values:
        ob1, ob2, delt1, delt2, st1, st2 = run_delete_combined_signed(seed, total_edges, n_val, k_val)

        E1 = ob_to_edges(genes_1, ob1)
        E2 = ob_to_edges(genes_2, ob2)
        E1_set = set(E1)
        E2_set = set(E2)

        row_1 = {"n_val": float(n_val), "k_val": float(k_val)}
        row_2 = {"n_val": float(n_val), "k_val": float(k_val)}
        for (e, col) in zip(initial_edges_1, edge_cols_1):
            row_1[col] = 1 if e in E1_set else 0
        for (e, col) in zip(initial_edges_2, edge_cols_2):
            row_2[col] = 1 if e in E2_set else 0
        rows_1.append(row_1)
        rows_2.append(row_2)

        typed_row_1 = {"n_val": float(n_val), "k_val": float(k_val)}
        typed_row_2 = {"n_val": float(n_val), "k_val": float(k_val)}
        typed_row_1.update(get_row_signs(ob1, genes_1, network_1, int_test_1, Part3_1[5], initial_edges_1))
        typed_row_2.update(get_row_signs(ob2, genes_2, network_2, int_test_2, Part3_2[5], initial_edges_2))
        typed_rows_1.append(typed_row_1)
        typed_rows_2.append(typed_row_2)

    df_1 = pd.DataFrame(rows_1, columns=["n_val", "k_val"] + edge_cols_1)
    df_2 = pd.DataFrame(rows_2, columns=["n_val", "k_val"] + edge_cols_2)
    df_1.to_csv(f"{out_dir}/auprc_table_1_signscore.csv", index=False)
    df_2.to_csv(f"{out_dir}/auprc_table_2_signscore.csv", index=False)

    tdf_1 = pd.DataFrame(typed_rows_1, columns=["n_val", "k_val"] + edge_cols_1)
    tdf_2 = pd.DataFrame(typed_rows_2, columns=["n_val", "k_val"] + edge_cols_2)
    tdf_1.to_csv(f"{out_dir}/auprc_table_1_signscore_typed.csv", index=False)
    tdf_2.to_csv(f"{out_dir}/auprc_table_2_signscore_typed.csv", index=False)

    return df_1, df_2, tdf_1, tdf_2


def export_AUPRC_table_base(seed, total_edges, CV_values):
    base = f"ODE_Case_{seed}"
    out_dir = f"{base}/total_{total_edges}_edges"

    genes_1 = list(pd.read_csv(f"{base}/data_1.csv", index_col=0).index.astype(str))
    genes_2 = list(pd.read_csv(f"{base}/data_2.csv", index_col=0).index.astype(str))

    network_1 = pd.read_csv(f"{out_dir}/initial_network_1.csv")
    network_2 = pd.read_csv(f"{out_dir}/initial_network_2.csv")
    network_1.columns = ["Source", "Target", "Interaction"]
    network_2.columns = ["Source", "Target", "Interaction"]
    network_1["Interaction"] = 1
    network_2["Interaction"] = 1

    consis_res_1 = joblib.load(f"{out_dir}/consis_1.joblib")
    consis_res_2 = joblib.load(f"{out_dir}/consis_2.joblib")
    int_test_1 = joblib.load(f"{out_dir}/int_test_1.joblib")
    int_test_2 = joblib.load(f"{out_dir}/int_test_2.joblib")
    Part3_1 = joblib.load(f"{out_dir}/Part3_1.joblib")
    Part3_2 = joblib.load(f"{out_dir}/Part3_2.joblib")

    initial_edges_1 = list(map(tuple, network_1[["Source", "Target"]].values))
    initial_edges_2 = list(map(tuple, network_2[["Source", "Target"]].values))
    edge_cols_1 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_1]
    edge_cols_2 = [f"{src} to {tgt}" for (src, tgt) in initial_edges_2]

    rows_1 = []
    rows_2 = []
    typed_rows_1 = []
    typed_rows_2 = []

    for CV in CV_values:
        delete_1 = NetDesDuo.delete_int(gene_list=genes_1, network=network_1, int_results=int_test_1,
                                     consis_res=consis_res_1, MSE_cut=CV)
        delete_2 = NetDesDuo.delete_int(gene_list=genes_2, network=network_2, int_results=int_test_2,
                                     consis_res=consis_res_2, MSE_cut=CV)

        E1 = ob_to_edges(genes_1, delete_1[2])
        E2 = ob_to_edges(genes_2, delete_2[2])
        E1_set = set(E1)
        E2_set = set(E2)

        row_1 = {"CV": float(CV)}
        row_2 = {"CV": float(CV)}
        for (e, col) in zip(initial_edges_1, edge_cols_1):
            row_1[col] = 1 if e in E1_set else 0
        for (e, col) in zip(initial_edges_2, edge_cols_2):
            row_2[col] = 1 if e in E2_set else 0
        rows_1.append(row_1)
        rows_2.append(row_2)

        typed_row_1 = {"CV": float(CV)}
        typed_row_2 = {"CV": float(CV)}
        typed_row_1.update(get_row_signs(delete_1[2], genes_1, network_1, int_test_1, Part3_1[5], initial_edges_1))
        typed_row_2.update(get_row_signs(delete_2[2], genes_2, network_2, int_test_2, Part3_2[5], initial_edges_2))
        typed_rows_1.append(typed_row_1)
        typed_rows_2.append(typed_row_2)

    df_1 = pd.DataFrame(rows_1, columns=["CV"] + edge_cols_1)
    df_2 = pd.DataFrame(rows_2, columns=["CV"] + edge_cols_2)
    df_1.to_csv(f"{out_dir}/auprc_table_base_1.csv", index=False)
    df_2.to_csv(f"{out_dir}/auprc_table_base_2.csv", index=False)

    tdf_1 = pd.DataFrame(typed_rows_1, columns=["CV"] + edge_cols_1)
    tdf_2 = pd.DataFrame(typed_rows_2, columns=["CV"] + edge_cols_2)
    tdf_1.to_csv(f"{out_dir}/auprc_table_base_1_typed.csv", index=False)
    tdf_2.to_csv(f"{out_dir}/auprc_table_base_2_typed.csv", index=False)

    return df_1, df_2, tdf_1, tdf_2

## Sweep the joint deletion

In [ ]:
#create AUPRC tables for combined deletion method
n_values = 	[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.8,1.0,1.2,1.5,2.0,2.5,3.0,3.5,4.0,5.0,6.0,8.0,10.0,15.0,20.0,50.0,100.0]
seeds = [125,177,629,753,1419, 1434, 1477,1594,1612,2120, 2569, 3219, 3646, 4119, 4250, 4510, 5042, 5236, 6072, 6534]
total_edges = [27, 30, 33, 36, 39, 42]
k_val = 1

tasks = [(s, total_edge) for s in seeds for total_edge in total_edges]
results = joblib.Parallel(n_jobs=20, backend="loky")(
    joblib.delayed(export_AUPRC_table)(s, total_edge, n_values, k_val) for (s, total_edge) in tasks
)
results = joblib.Parallel(n_jobs=20, backend="loky")(
    joblib.delayed(export_AUPRC_table_signed)(s, total_edge, n_values, k_val) for (s, total_edge) in tasks
)

## Sweep base NetDes

In [ ]:
#creates AUPRC tables for base NetDes method
CV_values = [10, 11, 12, 13, 14, 15, 17.5, 20, 22.5, 25, 27.5, 30, 32.5, 35, 37.5, 40, 42.5, 45, 47.5, 50, 55, 60, 65, 70, 75, 80, 
             85, 90, 95, 100, 110, 120, 130, 140, 150, 175, 200, 250, 300, 400, 500, 1000] 
seeds = [125,177,629,753,1419, 1434, 1477,1594,1612,2120, 2569, 3219, 3646, 4119, 4250, 4510, 5042, 5236, 6072, 6534]
total_edges = [27, 30, 33, 36, 39, 42]

tasks = [(s, total_edge) for s in seeds for total_edge in total_edges]
results = joblib.Parallel(n_jobs=10, backend="loky")(
    joblib.delayed(export_AUPRC_table_base)(s, total_edge, CV_values) for (s, total_edge) in tasks
)